

---

# **OpenAI Codex App Server: A Comprehensive Technical Report**

### **Unlocking the Codex Harness — Architecture, Protocol Design, and Multi-Surface Agent Integration**

**Reference Source:** *Celia Chen, Member of the Technical Staff, OpenAI*
**Report Classification:** Premium Technical Analysis — Agent Infrastructure & Distributed Systems

---

## **Table of Contents**

1. [Executive Summary](#1-executive-summary)
2. [Motivation & Architectural Genesis](#2-motivation--architectural-genesis)
3. [Codex Harness: Internal Architecture](#3-codex-harness-internal-architecture)
4. [App Server Process Architecture](#4-app-server-process-architecture)
5. [JSON-RPC Protocol: Formal Specification](#5-json-rpc-protocol-formal-specification)
6. [Conversation Primitives & State Machine Formalization](#6-conversation-primitives--state-machine-formalization)
7. [Client Integration Patterns](#7-client-integration-patterns)
8. [Transport Layer & Serialization Design](#8-transport-layer--serialization-design)
9. [Concurrency Model & Thread Lifecycle Management](#9-concurrency-model--thread-lifecycle-management)
10. [Protocol Selection Framework](#10-protocol-selection-framework)
11. [Design Principles & Engineering Lessons](#11-design-principles--engineering-lessons)
12. [Future Directions & Open Problems](#12-future-directions--open-problems)

---

## **1. Executive Summary**

OpenAI's **Codex** coding agent operates across a heterogeneous surface ecosystem — web application, command-line interface (CLI), IDE extensions (VS Code, JetBrains, Xcode), and a native macOS desktop application. Rather than reimplementing agent logic per surface, OpenAI engineered a **unified agent harness** exposed through the **Codex App Server**: a long-lived process communicating via a **bidirectional JSON-RPC 1.0 protocol over stdio (JSONL transport)**.

This report delivers an end-to-end technical dissection of:

- The internal composition of the Codex harness (agent loop, thread lifecycle, tool execution, authentication)
- The App Server's four-component process architecture (stdio reader, message processor, thread manager, core threads)
- The formal conversation primitive hierarchy: **Item** $\subset$ **Turn** $\subset$ **Thread**
- Three canonical client integration topologies: local apps/IDEs, web runtime, and TUI
- Protocol selection tradeoffs across JSON-RPC App Server, MCP server, cross-provider harness protocols, Codex Exec, and Codex SDK

---

## **2. Motivation & Architectural Genesis**

### **2.1 The Reuse Problem**

The Codex agent harness encapsulates the **core agent loop** — the orchestration logic governing the interaction cycle among user, model, and tools. Initially, the Codex CLI was built as a **TUI (Terminal User Interface)**, tightly coupling the agent loop to the terminal rendering layer.

When OpenAI developed the **VS Code extension**, a fundamental architectural requirement emerged:

> **Requirement:** Drive the identical agent loop from an IDE UI without reimplementing harness logic.

This demanded support for **rich interaction patterns** beyond simple request/response semantics:

| **Interaction Pattern** | **Description** |
|---|---|
| Workspace exploration | Agent reads and navigates project structure |
| Streaming progress | Real-time incremental updates during reasoning |
| Diff emission | Structured code modification artifacts |
| Approval gating | Human-in-the-loop permission flow for tool execution |
| Session persistence | Reconnection with consistent timeline rendering |

### **2.2 Why Not MCP?**

OpenAI initially experimented with exposing Codex as an **MCP (Model Context Protocol) server**. However, maintaining MCP semantics in a manner compatible with VS Code's interaction model proved architecturally untenable. The core friction points were:

1. **Session semantics mismatch:** MCP's tool-invocation model does not natively support the rich, stateful, multi-turn conversation lifecycle that Codex requires
2. **Streaming granularity:** MCP endpoints lack native support for fine-grained item-level delta streaming
3. **Approval flow:** Bidirectional server-initiated requests (e.g., tool approval) do not map cleanly to MCP's client→server invocation pattern

### **2.3 Evolution Trajectory**

The architectural evolution followed a clear progression:

$$
\text{TUI (monolithic)} \xrightarrow{\text{VS Code requirement}} \text{JSON-RPC v1 (ad-hoc)} \xrightarrow{\text{adoption pressure}} \text{App Server (stable platform API)}
$$

Key adoption pressure sources:

- **Internal teams:** Codex desktop app requiring parallel agent orchestration
- **External partners:** JetBrains, Xcode wanting IDE-grade agent experiences
- **Ecosystem demand:** Embedding the harness into third-party software development workflows

The resultant design constraints for the stable App Server:

| **Constraint** | **Rationale** |
|---|---|
| Easy to integrate | Minimize client-side implementation burden |
| Backward compatible | Protocol evolution must not break existing clients |
| Bidirectional | Server must initiate requests (approvals, input solicitation) |
| Streaming-native | Incremental updates are first-class, not bolted-on |
| Language-agnostic | Clients implemented in Go, Python, TypeScript, Swift, Kotlin |

---

## **3. Codex Harness: Internal Architecture**

The **Codex harness** encompasses the complete agent experience, decomposable into four functional subsystems:

### **3.1 Core Agent Loop**

The agent loop orchestrates the fundamental interaction cycle:

$$
\text{UserInput} \xrightarrow{\text{submit}} \text{Model Inference} \xrightarrow{\text{tool calls}} \text{Tool Execution} \xrightarrow{\text{observation}} \text{Model Inference} \xrightarrow{\text{response}} \text{AgentOutput}
$$

This loop may iterate multiple times within a single turn, as the model chains tool calls before producing a final response. Formally, a single turn can be modeled as:

$$
T = \langle m_0, (t_1, o_1), (t_2, o_2), \ldots, (t_k, o_k), m_f \rangle
$$

where:
- $m_0$ = initial user message
- $(t_i, o_i)$ = tool call and observation pair at step $i$
- $m_f$ = final agent message
- $k \geq 0$ = number of tool invocations

### **3.2 Thread Lifecycle & Persistence**

A **thread** is a durable conversation container with the following lifecycle operations:

| **Operation** | **Semantics** |
|---|---|
| `create` | Initialize a new thread with fresh state |
| `resume` | Reconnect to an existing thread, replaying event history |
| `fork` | Branch from an existing thread at a specific point |
| `archive` | Persist and close a thread for future reference |

Thread persistence guarantees that **event history is durable**, enabling clients to reconnect and render a **consistent timeline** without state reconstruction.

### **3.3 Configuration & Authentication**

The harness manages:

- **Configuration loading:** Default values, user overrides, workspace-specific settings
- **Authentication flows:** "Sign in with ChatGPT" OAuth-style flows, including credential state management and token refresh
- **Model discovery:** Enumerating available models and their capabilities

### **3.4 Tool Execution & Extensions**

Tool execution operates within a **sandbox** with a **consistent policy model**:

- **Shell/file tools:** Command execution, file read/write operations
- **MCP server integrations:** External tool servers wired into the agent loop
- **Skills:** Reusable, composable agent capabilities (e.g., `codex-skills`)

All tool executions are subject to the **approval policy**, which determines whether a tool call requires explicit user consent before execution.

### **3.5 Codebase Organization: Codex Core**

All agent logic resides in **`codex-core`** within the Codex CLI repository. `codex-core` serves a dual role:

| **Role** | **Description** |
|---|---|
| **Library** | Exposes agent code as importable Rust modules |
| **Runtime** | Can be instantiated to run the agent loop and manage persistence for one thread |

---

## **4. App Server Process Architecture**

The App Server is both a **protocol specification** and a **long-lived process** that hosts Codex core thread instances.

### **4.1 Four-Component Architecture**

The App Server process decomposes into four principal components:

```
┌─────────────────────────────────────────────────────────────────┐
│                      APP SERVER PROCESS                         │
│                                                                 │
│  ┌──────────┐    ┌─────────────────────┐    ┌────────────────┐  │
│  │  STDIO   │───▶│  CODEX MESSAGE      │───▶│   THREAD       │  │
│  │  READER  │    │  PROCESSOR          │    │   MANAGER      │  │
│  │          │◀───│                     │◀───│                │  │
│  └──────────┘    └─────────────────────┘    └───────┬────────┘  │
│                                                     │           │
│                              ┌───────────┬──────────┼────────┐  │
│                              ▼           ▼          ▼        │  │
│                         ┌────────┐  ┌────────┐  ┌────────┐   │  │
│                         │ CORE   │  │ CORE   │  │ CORE   │   │  │
│                         │THREAD 1│  │THREAD 2│  │THREAD N│   │  │
│                         └────────┘  └────────┘  └────────┘   │  │
│                                                              │  │
└─────────────────────────────────────────────────────────────────┘
```

#### **Component Responsibilities**

| **Component** | **Responsibility** |
|---|---|
| **Stdio Reader** | Reads JSONL-encoded JSON-RPC messages from `stdin`, parses and validates framing |
| **Codex Message Processor** | Translates JSON-RPC requests → Codex core operations; transforms core events → JSON-RPC notifications |
| **Thread Manager** | Lifecycle manager for core threads; spins up one core session per thread; handles lookup, creation, and teardown |
| **Core Threads** | Individual `codex-core` runtime instances; each manages the agent loop and persistence for a single conversation |

### **4.2 Data Flow**

The data flow through the App Server follows a **bidirectional pipeline**:

**Client → Server (Request Path):**

$$
\text{Client} \xrightarrow{\text{JSON-RPC}} \text{Stdio Reader} \xrightarrow{\text{parse}} \text{Message Processor} \xrightarrow{\text{lookup}} \text{Thread Manager} \xrightarrow{\text{submit}} \text{Core Thread}
$$

**Server → Client (Event Path):**

$$
\text{Core Thread} \xrightarrow{\text{internal events}} \text{Message Processor} \xrightarrow{\text{transform}} \text{Stdio Writer} \xrightarrow{\text{JSON-RPC notification}} \text{Client}
$$

### **4.3 Translation Layer Design**

The **Stdio Reader + Message Processor** pair serves as a critical **translation layer** with two functions:

1. **Inbound translation:** Client JSON-RPC requests $\mapsto$ Codex core operations
2. **Outbound translation:** Codex core internal event stream $\mapsto$ stable, UI-ready JSON-RPC notifications

This translation layer is architecturally essential because it **decouples the internal event representation** (which may evolve rapidly) from the **stable external API surface** (which must maintain backward compatibility).

### **4.4 Multiplexing Model**

The relationship between clients and core threads follows a **1:N multiplexing** pattern:

$$
\text{1 App Server process} : N \text{ Core Threads} : 1 \text{ Client Connection}
$$

One client request can produce **many event updates**, enabling rich UI rendering:

$$
|\text{notifications}(r)| \gg 1 \quad \forall \; r \in \text{Requests}
$$

This is the fundamental architectural property that enables streaming progress, incremental diff updates, and multi-step tool execution visibility.

---

## **5. JSON-RPC Protocol: Formal Specification**

### **5.1 Protocol Foundation**

The App Server uses **JSON-RPC 1.0** over **stdio** with **JSONL (newline-delimited JSON)** framing. The protocol is **fully bidirectional**:

| **Direction** | **Message Type** | **Example** |
|---|---|---|
| Client → Server | Request | `initialize`, `thread/start`, `turn/start` |
| Server → Client | Response | Result for `initialize` |
| Server → Client | Notification | `thread/started`, `item/started`, `item/*/delta`, `item/completed` |
| Server → Client | Request (server-initiated) | `item/commandExecution/requestApproval` |
| Client → Server | Response (to server request) | Approval decision (`allow` / `deny`) |

### **5.2 Initialization Handshake**

The protocol mandates a **single `initialize` request** before any other method. This handshake serves three purposes:

1. **Capability advertisement:** Server declares supported features
2. **Protocol versioning:** Both sides agree on version and feature flags
3. **Default negotiation:** Establish configuration defaults

**Client → Server:**

```json
{
  "method": "initialize",
  "id": 0,
  "params": {
    "clientInfo": {
      "name": "codex_vscode",
      "title": "Codex VS Code Extension",
      "version": "0.1.0"
    }
  }
}
```

**Server → Client:**

```json
{
  "id": 0,
  "result": {
    "userAgent": "codex_vscode/0.94.0-alpha.7 (Mac OS 26.2.0; arm64) vscode/2.4.22 (codex_vscode; 0.1.0)"
  }
}
```

The `userAgent` string encodes:
- App Server version (`0.94.0-alpha.7`)
- Platform (`Mac OS 26.2.0; arm64`)
- Client identifier and version (`vscode/2.4.22`, `codex_vscode/0.1.0`)

### **5.3 Message Taxonomy**

The complete message taxonomy can be formalized as:

$$
\mathcal{M} = \mathcal{M}_{\text{req}}^{C \to S} \cup \mathcal{M}_{\text{res}}^{S \to C} \cup \mathcal{M}_{\text{notif}}^{S \to C} \cup \mathcal{M}_{\text{req}}^{S \to C} \cup \mathcal{M}_{\text{res}}^{C \to S}
$$

where each subset represents a specific direction and message type in the bidirectional protocol.

---

## **6. Conversation Primitives & State Machine Formalization**

### **6.1 Primitive Hierarchy**

The App Server protocol defines a **three-level primitive hierarchy** with strict containment:

$$
\text{Item} \subset \text{Turn} \subset \text{Thread}
$$

```
Thread (durable conversation container)
├── Turn₁ (one unit of agent work)
│   ├── Item₁ (user message)
│   ├── Item₂ (tool execution)
│   ├── Item₃ (approval request)
│   ├── Item₄ (diff)
│   └── Item₅ (agent message)
├── Turn₂
│   ├── Item₆
│   └── Item₇
└── ...
```

### **6.2 Item: Atomic Unit**

An **Item** is the atomic unit of input/output. Items are **typed** and follow an explicit **lifecycle state machine**:

$$
\text{Item FSM}: \quad \boxed{\text{started}} \xrightarrow{\text{delta}^*} \boxed{\text{completed}}
$$

| **Event** | **Semantics** | **Cardinality** |
|---|---|---|
| `item/started` | Item begins; client can start rendering | Exactly 1 |
| `item/*/delta` | Incremental content update (streaming) | 0 or more |
| `item/completed` | Item finalizes with terminal payload | Exactly 1 |

**Item Types:**

| **Type** | **Description** | **Supports Delta** |
|---|---|---|
| `user_message` | User input text | No |
| `agent_message` | Agent response text | Yes (streaming) |
| `tool_execution` | Shell/file tool invocation | Yes (output streaming) |
| `approval_request` | Server requests human approval | No |
| `diff` | Code modification artifact | Yes (incremental hunks) |

Formally, the lifecycle of a single item $i$ can be represented as an ordered event sequence:

$$
\mathcal{E}(i) = \langle e_{\text{started}}, \underbrace{e_{\delta_1}, e_{\delta_2}, \ldots, e_{\delta_m}}_{\text{optional streaming deltas}}, e_{\text{completed}} \rangle
$$

where $m \geq 0$ and the total content at completion satisfies:

$$
\text{content}(e_{\text{completed}}) = \bigoplus_{j=1}^{m} \text{content}(e_{\delta_j})
$$

with $\bigoplus$ denoting the appropriate concatenation operator for the item's content type.

### **6.3 Turn: Unit of Agent Work**

A **Turn** represents one unit of agent work initiated by user input. It encapsulates the complete sequence of items produced in response to a single user submission.

$$
\text{Turn FSM}: \quad \boxed{\text{started}} \xrightarrow{\text{items}} \boxed{\text{completed}}
$$

Formally:

$$
\text{Turn} = \langle \text{user\_input}, [I_1, I_2, \ldots, I_n] \rangle
$$

where each $I_j$ is an Item following the lifecycle defined above, and $n \geq 1$.

**Important property:** A turn may be **paused** mid-execution when the server issues an approval request. The turn resumes only after the client responds:

$$
\text{Turn with approval}: \quad \boxed{\text{started}} \xrightarrow{\text{items}} \boxed{\text{paused (awaiting approval)}} \xrightarrow{\text{client response}} \boxed{\text{resumed}} \xrightarrow{\text{items}} \boxed{\text{completed}}
$$

### **6.4 Thread: Durable Container**

A **Thread** is the top-level durable container with full lifecycle management:

$$
\text{Thread FSM}: \quad \boxed{\text{created}} \xrightarrow{\text{turns}} \boxed{\text{active}} \xrightarrow{\text{archive}} \boxed{\text{archived}}
$$

Extended operations:

$$
\boxed{\text{active}} \xrightarrow{\text{fork}} \boxed{\text{active (new branch)}}
$$

$$
\boxed{\text{archived}} \xrightarrow{\text{resume}} \boxed{\text{active}}
$$

Thread persistence guarantees:

$$
\forall \; \text{thread} \; T, \quad \text{history}(T) = \bigoplus_{k=1}^{|\text{turns}|} \mathcal{E}(\text{Turn}_k) \quad \text{is durable and replayable}
$$

### **6.5 Complete Protocol Flow: Annotated Sequence**

The following represents a full conversation lifecycle with all primitive interactions:

```
CLIENT                                           SERVER
  │                                                │
  │──── initialize {clientInfo} ──────────────────▶│
  │◀─── result {userAgent} ───────────────────────│
  │                                                │
  │──── thread/start ─────────────────────────────▶│
  │◀─── thread/started ──────────────────────────│
  │                                                │
  │──── turn/start {input: "run tests..."} ──────▶│
  │◀─── turn/started ────────────────────────────│
  │◀─── item/started {type: user_message} ───────│
  │◀─── item/completed {content: "run tests..."}─│
  │                                                │
  │◀─── item/started {type: tool_execution} ─────│
  │◀─── item/commandExecution/requestApproval ───│  ← SERVER-INITIATED REQUEST
  │──── approval {decision: "allow"} ────────────▶│  ← CLIENT RESPONSE
  │◀─── item/completed {cmd: "pnpm test"} ───────│
  │                                                │
  │◀─── item/started {type: agent_message} ──────│
  │◀─── item/agentMessage/delta {"ran 3 tests."} │
  │◀─── item/agentMessage/delta {"all passed"} ──│
  │◀─── item/completed {full message} ───────────│
  │                                                │
  │◀─── turn/completed ──────────────────────────│
  │                                                │
```

### **6.6 Debugging & Inspection**

The App Server provides a built-in test client for protocol inspection:

```bash
codex debug app-server send-message-v2 "run tests and summarize failures"
```

This command executes a full turn and outputs the raw JSON-RPC message sequence, enabling protocol-level debugging without building a custom client.

---

## **7. Client Integration Patterns**

The App Server supports three canonical integration topologies, each optimized for a different deployment context.

### **7.1 Pattern 1: Local Apps & IDEs**

**Applicable Surfaces:** VS Code Extension, JetBrains IDEs, Xcode, Codex Desktop App

**Architecture:**

```
┌──────────────────┐     stdio (JSONL)     ┌──────────────────┐
│   IDE / Desktop  │◀────────────────────▶│   App Server     │
│   Application    │    JSON-RPC bidir     │   (child process)│
│                  │                       │                  │
│   [Client Code]  │                       │  [Codex Core]    │
└──────────────────┘                       └──────────────────┘
```

**Key Characteristics:**

| **Aspect** | **Detail** |
|---|---|
| **Binary distribution** | Platform-specific App Server binary bundled with or fetched by the client |
| **Process model** | App Server launched as a **long-running child process** |
| **Transport** | Bidirectional stdio channel (JSONL) |
| **Version pinning** | Client ships pinned to a tested App Server version |
| **Decoupled releases** | Partners (e.g., Xcode) can point to newer server binaries without client updates |

**Backward Compatibility Guarantee:**

$$
\forall \; v_{\text{client}} \leq v_{\text{server}}, \quad \text{protocol}(v_{\text{client}}, v_{\text{server}}) \text{ is valid}
$$

Older clients can communicate with newer servers safely. This property is enforced by the App Server's JSON-RPC surface design.

### **7.2 Pattern 2: Codex Web Runtime**

**Applicable Surfaces:** Codex Web Application (browser-based)

**Architecture:**

```
┌──────────┐   HTTP/SSE    ┌──────────────┐   stdio    ┌──────────────┐
│  Browser  │◀────────────▶│   Codex      │◀──────────▶│  App Server  │
│  (Web UI) │              │   Backend    │  JSON-RPC  │  (in container)│
│           │              │   (Worker)   │            │              │
└──────────┘              └──────────────┘            └──────────────┘
                                │
                                ▼
                     ┌──────────────────┐
                     │  Provisioned     │
                     │  Container with  │
                     │  Workspace       │
                     └──────────────────┘
```

**Key Characteristics:**

| **Aspect** | **Detail** |
|---|---|
| **Environment** | Containerized; workspace checked out inside container |
| **Server-side state** | Worker maintains state and progress; browser is stateless |
| **Browser transport** | HTTP + Server-Sent Events (SSE) for streaming |
| **Internal transport** | JSON-RPC over stdio between worker and App Server |
| **Ephemeral clients** | Browser tabs may close; work continues server-side |
| **Reconnection** | New sessions catch up via persisted thread history |

**Design Rationale:**

Web sessions are inherently ephemeral. The architectural principle is:

> **The browser is never the source of truth for long-running tasks.**

By keeping state on the server:
- Work continues if the tab closes or the network drops
- Reconnection is trivial via thread history replay
- No client-side state reconstruction is necessary

### **7.3 Pattern 3: TUI / Codex CLI**

**Applicable Surfaces:** Terminal-based Codex interface

**Current State (Legacy):**

The TUI historically runs as a **"native" client** in the same process as the agent loop, directly invoking Rust core types rather than communicating through the App Server protocol. This was an expedient architectural choice for rapid iteration but creates a **special-case surface** that does not benefit from the stable protocol layer.

**Planned Refactor:**

```
┌──────────┐     stdio (JSONL)     ┌──────────────────┐
│   TUI    │◀────────────────────▶│   App Server     │
│  (Rust)  │    JSON-RPC bidir     │   (child process)│
└──────────┘                       └──────────────────┘
```

The refactor will make the TUI behave like any other client, unlocking:

| **Capability** | **Description** |
|---|---|
| **Remote execution** | TUI connects to App Server on a remote machine |
| **Compute proximity** | Agent stays close to compute (GPU, workspace) |
| **Disconnection resilience** | Work continues if laptop sleeps or disconnects |
| **Live updates** | Local TUI receives streaming events from remote server |

### **7.4 Client Language Implementations**

The JSON-RPC protocol's language-agnosticism has enabled client implementations across:

| **Language** | **Client** |
|---|---|
| TypeScript | VS Code Extension, Codex SDK |
| Swift | Xcode integration |
| Kotlin | JetBrains IDEs |
| Go | Internal tooling |
| Python | Internal tooling, automation |

**Type Generation Utilities:**

```bash
# Generate TypeScript definitions from Rust protocol types
codex app-server generate-ts

# Generate JSON Schema bundle for any language's code generator
codex app-server generate-json-schema
```

---

## **8. Transport Layer & Serialization Design**

### **8.1 Transport: Stdio with JSONL Framing**

The transport choice of **stdio** (standard input/output) with **JSONL** (JSON Lines) framing is a deliberate architectural decision:

| **Property** | **Benefit** |
|---|---|
| **Universal availability** | Every OS and language runtime supports stdio |
| **No network stack** | Eliminates TCP/HTTP overhead, TLS configuration, port management |
| **Process isolation** | App Server runs as child process with natural lifecycle management |
| **Simplicity** | One line = one JSON-RPC message; trivial to parse and debug |
| **Cross-platform** | Identical transport semantics on macOS, Linux, Windows |

### **8.2 Why JSON-RPC?**

JSON-RPC was selected over alternatives for specific technical reasons:

| **Alternative** | **Reason for Rejection** |
|---|---|
| gRPC / Protobuf | Requires schema compilation, binary framing complexity, less human-readable |
| REST / HTTP | Not bidirectional without WebSocket; request/response model insufficient |
| MCP | Semantic mismatch for rich agent session lifecycle (see §2.2) |
| Custom binary protocol | Development overhead, debugging difficulty, no ecosystem tooling |

JSON-RPC provides:
- **Bidirectional messaging** (both client and server can initiate requests)
- **Correlation via `id` field** (match responses to requests)
- **Notification support** (fire-and-forget messages without response expectation)
- **Human-readable** (critical for debugging and protocol inspection)

### **8.3 Serialization Pipeline**

The serialization pipeline from Rust core types to wire format:

$$
\text{Rust structs} \xrightarrow{\text{serde}} \text{JSON} \xrightarrow{\text{newline framing}} \text{JSONL} \xrightarrow{\text{stdio write}} \text{Wire}
$$

The inverse path for deserialization:

$$
\text{Wire} \xrightarrow{\text{stdio read}} \text{JSONL} \xrightarrow{\text{line split}} \text{JSON} \xrightarrow{\text{serde}} \text{Rust structs}
$$

---

## **9. Concurrency Model & Thread Lifecycle Management**

### **9.1 Thread Manager: Core Session Multiplexing**

The **Thread Manager** is responsible for maintaining a mapping:

$$
\mathcal{T}: \text{ThreadID} \to \text{CoreThread}
$$

where each `CoreThread` is an independent `codex-core` runtime instance with its own:
- Agent loop state
- Conversation history
- Persistence store
- Tool execution sandbox

### **9.2 Parallelism Model**

The App Server supports **concurrent thread execution**, which is critical for surfaces like the Codex Desktop App that orchestrate multiple agents simultaneously:

$$
\text{App Server} \ni \{C_1, C_2, \ldots, C_N\} \quad \text{where each } C_i \text{ executes independently}
$$

The concurrency model leverages Rust's async runtime (likely `tokio`) for:
- Non-blocking I/O on the stdio channel
- Concurrent event processing across multiple core threads
- Backpressure management when event production exceeds client consumption rate

### **9.3 Event Fan-Out**

For a single client request $r$ targeting thread $T_k$:

$$
r \xrightarrow{\text{submit to } C_k} \{e_1, e_2, \ldots, e_p\} \quad \text{where } p \gg 1
$$

The Message Processor aggregates events from all active core threads and serializes them onto the single stdio output channel. This requires a **merge strategy** to interleave events from concurrent threads:

$$
\text{output stream} = \text{merge}(\mathcal{E}(C_1), \mathcal{E}(C_2), \ldots, \mathcal{E}(C_N))
$$

Each event is tagged with its originating thread ID, enabling the client to demultiplex and route events to the appropriate UI context.

---

## **10. Protocol Selection Framework**

### **10.1 Comparison Matrix**

| **Method** | **Protocol** | **Bidirectional** | **Streaming** | **Full Harness** | **Best For** |
|---|---|---|---|---|---|
| **Codex App Server** | JSON-RPC / stdio | ✅ | ✅ | ✅ | IDE integrations, desktop apps, full agent UIs |
| **Codex MCP Server** | MCP / stdio | Partial | Limited | ❌ | MCP-based workflows, tool invocation |
| **Cross-Provider Harness** | Varies | Varies | Varies | ❌ | Multi-provider agent coordination |
| **Codex Exec** | CLI (stdout) | ❌ | ✅ (structured) | ❌ | CI/CD, automation, one-off tasks |
| **Codex SDK** | TypeScript API | ❌ | ✅ | Partial | Programmatic control, server-side tools |

### **10.2 Decision Framework**

The protocol selection can be formalized as an optimization over requirements:

$$
\text{protocol}^* = \arg\max_{p \in \mathcal{P}} \sum_{i} w_i \cdot \text{score}(p, r_i)
$$

where $\mathcal{P}$ = set of available protocols, $r_i$ = requirement $i$, and $w_i$ = importance weight.

**Decision tree:**

```
Need full agent UI with streaming + approvals?
├── YES → Codex App Server
└── NO
    ├── Already have MCP workflow?
    │   ├── YES → Codex MCP Server
    │   └── NO
    │       ├── CI/CD or one-off automation?
    │       │   ├── YES → Codex Exec
    │       │   └── NO
    │       │       ├── TypeScript programmatic control?
    │       │       │   ├── YES → Codex SDK
    │       │       │   └── NO → Codex App Server
    │       │       └──
    │       └──
    └──
```

### **10.3 Codex App Server: Detailed Capabilities**

| **Capability** | **Description** |
|---|---|
| Full agent loop | Complete reasoning → tool → response cycle |
| Sign in with ChatGPT | Built-in authentication flow |
| Model discovery | Enumerate available models and capabilities |
| Configuration management | Load, override, and manage settings |
| Thread persistence | Durable conversation history |
| Approval flow | Human-in-the-loop tool execution gating |
| Streaming events | Fine-grained item-level delta streaming |
| Backward compatibility | Older clients work with newer servers |

### **10.4 Codex Exec: Scriptable CLI Mode**

```bash
codex exec "run tests and fix failures"
```

| **Property** | **Value** |
|---|---|
| Interaction mode | Non-interactive, runs to completion |
| Output format | Structured (parseable for logs) |
| Exit semantics | Clear success/failure signal |
| Use case | CI pipelines, batch automation, scheduled tasks |

### **10.5 Codex SDK: TypeScript Library**

```typescript
import { CodexAgent } from '@openai/codex-sdk';

const agent = new CodexAgent({ model: 'gpt-5.2-codex' });
const result = await agent.run('refactor auth module');
```

| **Property** | **Value** |
|---|---|
| Language | TypeScript (currently) |
| Interface | Native library API |
| Use case | Server-side tools, programmatic workflows |
| Relationship to App Server | Predates App Server; smaller surface area |

---

## **11. Design Principles & Engineering Lessons**

### **11.1 Principle: Protocol as Platform**

The evolution from ad-hoc internal API to stable platform surface follows a well-known pattern in distributed systems:

$$
\text{Internal tool} \xrightarrow{\text{adoption}} \text{Informal API} \xrightarrow{\text{stability demand}} \text{Platform protocol}
$$

**Key lesson:** Design for stability early even when you don't expect external dependents. The cost of retroactive stabilization far exceeds the cost of proactive protocol design.

### **11.2 Principle: Translation Layer Isolation**

The Stdio Reader + Message Processor translation layer provides a critical **impedance mismatch buffer**:

$$
\text{Internal representation (volatile)} \xleftrightarrow{\text{translation}} \text{External API (stable)}
$$

This allows the Codex core internal event format to evolve freely without breaking external clients, as long as the translation layer maps between them correctly.

### **11.3 Principle: Primitives Over Procedures**

By designing around three primitives (Item, Turn, Thread) rather than a flat set of RPC methods, the protocol achieves:

1. **Composability:** Complex interactions are built from simple, well-defined building blocks
2. **Predictability:** Clients know exactly what lifecycle events to expect for each primitive
3. **Extensibility:** New item types can be added without changing the protocol structure

### **11.4 Principle: Streaming as First-Class**

The `item/*/delta` event pattern makes streaming a **first-class protocol citizen** rather than an afterthought:

- Clients can begin rendering on `item/started`
- Progressive refinement through delta events
- Finalization guarantees on `item/completed`

This design eliminates the common antipattern of retrofitting streaming onto a request/response protocol.

### **11.5 Principle: Server-Initiated Requests for Human-in-the-Loop**

The bidirectional nature of JSON-RPC enables the **approval flow** pattern, which is critical for safe agent operation:

$$
\text{Server} \xrightarrow{\text{requestApproval}} \text{Client} \xrightarrow{\text{allow/deny}} \text{Server}
$$

This pattern generalizes to any scenario where the agent needs human input mid-execution, supporting the broader principle of **human oversight in agentic systems**.

### **11.6 Principle: Meta-Integration via Self-Hosting**

OpenAI reports that many partner teams used **Codex itself** to build their App Server integrations:

> *"Many teams we worked with were able to make a working integration quickly using Codex."*

This self-referential property — where the agent assists in building its own client integrations — creates a virtuous cycle:

$$
\text{Better Codex} \to \text{Easier integration} \to \text{More clients} \to \text{More feedback} \to \text{Better Codex}
$$

---

## **12. Future Directions & Open Problems**

### **12.1 TUI Refactor to App Server Client**

The planned TUI refactor will eliminate the last "special-case" surface, achieving **full protocol unification**:

$$
\text{Surfaces} = \{\text{Web, VS Code, JetBrains, Xcode, Desktop, TUI}\} \quad \forall s \in \text{Surfaces}: s \text{ uses App Server}
$$

This unlocks remote execution workflows where the TUI serves as a thin rendering client while the App Server runs on a remote machine with access to compute and workspace resources.

### **12.2 Additional SDK Language Support**

Based on developer interest, OpenAI may develop SDKs that wrap the App Server protocol for additional languages, eliminating the need for teams to build raw JSON-RPC bindings.

### **12.3 Emerging Standards for Agent Protocols**

The report acknowledges that the agent protocol space is rapidly evolving. Skills represent one example of primitives that may converge into cross-provider standards. The key open question:

> **What is the minimal set of primitives that captures the full richness of real-world agent workflows while remaining portable across providers?**

### **12.4 Scalability Challenges**

As Codex adoption grows, several scalability challenges emerge:

| **Challenge** | **Description** |
|---|---|
| **Thread density** | Maximum concurrent core threads per App Server process |
| **Event throughput** | Sustaining high-frequency delta streams under load |
| **Persistence I/O** | Thread history write amplification with long conversations |
| **Auto-compaction** | Intelligent context window management within core threads |

---

## **Appendix A: Quick Reference — Key Commands**

| **Command** | **Purpose** |
|---|---|
| `codex debug app-server send-message-v2 "<prompt>"` | Run a full turn and inspect raw JSON-RPC messages |
| `codex app-server generate-ts` | Generate TypeScript type definitions from Rust protocol |
| `codex app-server generate-json-schema` | Generate JSON Schema bundle for code generation |
| `codex mcp-server` | Launch Codex as an MCP server |
| `codex exec "<prompt>"` | Run a one-off non-interactive task |

---

## **Appendix B: Protocol Message Reference**

| **Method** | **Direction** | **Type** | **Description** |
|---|---|---|---|
| `initialize` | $C \to S$ | Request | Handshake with client info |
| `thread/start` | $C \to S$ | Request | Create a new thread |
| `thread/started` | $S \to C$ | Notification | Thread creation confirmed |
| `turn/start` | $C \to S$ | Request | Submit user input, begin turn |
| `turn/started` | $S \to C$ | Notification | Turn execution begun |
| `turn/completed` | $S \to C$ | Notification | Turn execution finished |
| `item/started` | $S \to C$ | Notification | New item begun |
| `item/*/delta` | $S \to C$ | Notification | Incremental content update |
| `item/completed` | $S \to C$ | Notification | Item finalized |
| `item/commandExecution/requestApproval` | $S \to C$ | Request | Server requests tool approval |

---

## **Appendix C: Glossary**

| **Term** | **Definition** |
|---|---|
| **Codex Harness** | The complete agent logic including core loop, lifecycle management, auth, and tool execution |
| **Codex Core** | The Rust library/runtime in the Codex CLI repo containing all agent code |
| **App Server** | Long-lived process exposing Codex Core via bidirectional JSON-RPC over stdio |
| **Thread** | Durable conversation container with persistence and lifecycle management |
| **Turn** | One unit of agent work initiated by user input |
| **Item** | Atomic unit of I/O with typed content and explicit lifecycle |
| **JSONL** | JSON Lines — newline-delimited JSON encoding |
| **MCP** | Model Context Protocol — an alternative tool invocation protocol |
| **TUI** | Terminal User Interface — terminal-based Codex access |
| **Skills** | Reusable, composable agent capabilities |

---

**Repository:** [Codex CLI (Open Source)](https://github.com/openai/codex)

---

*End of Technical Report*



# OpenAI Codex App Server: A Comprehensive Technical Report

## Architectural Design, Protocol Engineering, and Multi-Surface Agent Integration

---

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Motivation and Historical Context](#2-motivation-and-historical-context)
3. [The Codex Harness: Internal Architecture](#3-the-codex-harness-internal-architecture)
4. [App Server Process Architecture](#4-app-server-process-architecture)
5. [JSON-RPC Protocol Design and Bidirectional Communication](#5-json-rpc-protocol-design-and-bidirectional-communication)
6. [Conversation Primitives: Formal Specification](#6-conversation-primitives-formal-specification)
7. [Protocol Message Flow: Detailed Walkthrough](#7-protocol-message-flow-detailed-walkthrough)
8. [Multi-Surface Client Integration Patterns](#8-multi-surface-client-integration-patterns)
9. [Protocol Comparison and Selection Framework](#9-protocol-comparison-and-selection-framework)
10. [Supplementary Embedding Mechanisms](#10-supplementary-embedding-mechanisms)
11. [Systems Engineering Considerations](#11-systems-engineering-considerations)
12. [Formal Analysis and Design Rationale](#12-formal-analysis-and-design-rationale)
13. [Conclusion and Forward Trajectory](#13-conclusion-and-forward-trajectory)

---

## 1. Executive Summary

OpenAI's **Codex** is a production-grade autonomous coding agent capable of reasoning over codebases, executing shell and file operations, producing structured diffs, and engaging in multi-turn interactive sessions with human developers. It is deployed across five distinct surfaces: the **web application**, the **CLI/TUI**, the **VS Code IDE extension**, **third-party IDE integrations** (JetBrains, Xcode), and the **macOS desktop application**.

The central engineering challenge addressed in this report is the **unification problem**: how to expose a single, deterministic agent loop—encompassing model inference, tool execution, approval gating, thread persistence, and streaming output—to heterogeneous client environments that differ in language runtime, UI paradigm, deployment topology, and release cadence.

The solution is the **Codex App Server**, a long-lived process that hosts the Codex harness and exposes it via a **bidirectional JSON-RPC 2.0 protocol** over **stdio (JSONL)**. This report provides an exhaustive technical dissection of its architecture, protocol semantics, conversation primitives, integration patterns, and design tradeoffs.

---

## 2. Motivation and Historical Context

### 2.1 The Reuse Imperative

The Codex agent loop is non-trivial. It orchestrates:

- **Model invocation** with structured tool-use schemas
- **Sandboxed tool execution** (shell commands, file I/O)
- **Human-in-the-loop approval gates**
- **Streaming output** with incremental deltas
- **Thread persistence** and resumption across sessions
- **Configuration management**, authentication, and credential lifecycle

Re-implementing this logic per surface would introduce:

$$
C_{\text{reimplementation}} = N_{\text{surfaces}} \times C_{\text{harness}} + \binom{N_{\text{surfaces}}}{2} \times C_{\text{divergence}}
$$

where $C_{\text{harness}}$ is the cost of implementing the full agent loop and $C_{\text{divergence}}$ captures the combinatorial cost of behavioral drift between independent implementations. With $N_{\text{surfaces}} \geq 5$, this cost becomes untenable.

### 2.2 Evolutionary Trajectory

The App Server's genesis followed a three-phase evolution:

| Phase | Surface | Architecture | Limitation |
|-------|---------|-------------|------------|
| **Phase 1** | CLI/TUI | In-process Rust integration; TUI directly calls Codex core types | Tightly coupled; no external client support |
| **Phase 2** | VS Code Extension | MCP server experiment | MCP semantics poorly mapped to IDE interaction patterns (streaming diffs, approval gates, thread lifecycle) |
| **Phase 3** | Multi-surface | JSON-RPC App Server over stdio | Stable, bidirectional, client-agnostic protocol |

The transition from Phase 2 to Phase 3 is architecturally significant. The **Model Context Protocol (MCP)** is optimized for tool-as-a-service invocations—stateless, request/response interactions where a host orchestrator calls a tool and receives a result. The Codex agent loop, however, requires:

- **Stateful thread management** across turns
- **Server-initiated requests** (approval gates)
- **Streaming multi-item output** within a single turn
- **Durable session semantics** (reconnection, fork, archive)

These requirements exceed MCP's design envelope, motivating a purpose-built protocol.

### 2.3 Design Requirements

From the accumulated experience, the following non-negotiable requirements were established:

1. **Bidirectionality**: Both client and server must be able to initiate requests.
2. **Streaming**: Items must support incremental delta delivery.
3. **Lifecycle explicitness**: Every entity (thread, turn, item) must have well-defined start/complete boundaries.
4. **Backward compatibility**: Protocol evolution must not break existing clients.
5. **Transport simplicity**: The protocol must work over stdio to support child-process spawning without network configuration.
6. **Language agnosticism**: Clients in Go, Python, TypeScript, Swift, Kotlin, and Rust must be viable without bespoke SDKs.

---

## 3. The Codex Harness: Internal Architecture

### 3.1 Codex Core: Library and Runtime

The **Codex core** resides in the Codex CLI codebase and serves a dual role:

- **As a library**: Provides the agent loop logic, tool execution engine, configuration resolution, and thread persistence as composable Rust modules.
- **As a runtime**: Can be instantiated to manage a single Codex thread (conversation), running the agent loop to completion or until a turn boundary.

### 3.2 Functional Decomposition

The harness decomposes into three principal subsystems:

#### 3.2.1 Thread Lifecycle and Persistence

A **thread** $\mathcal{T}$ is defined as a durable, ordered sequence of turns:

$$
\mathcal{T} = \langle t_1, t_2, \ldots, t_n \rangle
$$

Each turn $t_i$ contains an ordered sequence of items:

$$
t_i = \langle I_{i,1}, I_{i,2}, \ldots, I_{i,m_i} \rangle
$$

Thread operations include:

| Operation | Semantics |
|-----------|-----------|
| `create` | Instantiate a new thread with empty history |
| `resume` | Reconnect to an existing thread, replaying persisted event history |
| `fork` | Clone a thread at a specific turn boundary, creating a divergent timeline |
| `archive` | Mark a thread as inactive, preserving its history for later retrieval |

Persistence ensures that clients can disconnect and reconnect without state loss. The event history is the source of truth; UI state is a projection of this history.

#### 3.2.2 Configuration and Authentication

Codex manages:

- **Model selection**: Resolution of model identifiers (e.g., `gpt-5.2-codex`) to API endpoints
- **Authentication flows**: "Sign in with ChatGPT" credential lifecycle, token refresh, and credential state propagation to clients
- **Configuration defaults**: Workspace-level, user-level, and session-level configuration merging with well-defined precedence

#### 3.2.3 Tool Execution and Extensions

Tool execution operates under a **policy model** that governs:

- **Sandboxing**: Shell and file tools execute in isolated environments with controlled filesystem access
- **MCP server integration**: External MCP servers can be wired into the agent loop, participating as tool providers under the same approval and policy framework
- **Skills**: Reusable, composable capability definitions that extend the agent's action space

The tool execution engine implements the following control flow:

$$
\text{ToolCall} \xrightarrow{\text{policy check}} \begin{cases} \text{Auto-approve} & \text{if policy allows} \\ \text{Request approval} & \text{if policy requires human confirmation} \\ \text{Deny} & \text{if policy prohibits} \end{cases}
$$

---

## 4. App Server Process Architecture

### 4.1 Component Topology

The App Server is a **long-lived process** with four principal components arranged in a pipeline:

```
┌─────────┐    ┌───────────────────┐    ┌─────────────────┐    ┌──────────────┐
│  stdio   │───▶│  Codex Message    │───▶│  Thread Manager  │───▶│ Core Thread  │
│  Reader  │◀───│  Processor        │◀───│                  │◀───│  (per-thread)│
└─────────┘    └───────────────────┘    └─────────────────┘    └──────────────┘
     ▲                                                                │
     │                    JSON-RPC (JSONL over stdio)                 │
     ▼                                                                ▼
 ┌────────┐                                                   ┌──────────────┐
 │ Client │                                                   │ Core Thread  │
 └────────┘                                                   │  (per-thread)│
                                                              └──────────────┘
```

#### 4.1.1 stdio Reader

- **Function**: Reads newline-delimited JSON (JSONL) from stdin, parses each line as a JSON-RPC 2.0 message, and dispatches to the Codex Message Processor.
- **Design choice**: stdio was selected over TCP/HTTP for several reasons:
  - No port conflicts or firewall configuration
  - Natural child-process lifecycle management (parent death closes pipes)
  - Universal OS support
  - Deterministic startup ordering (no connection race conditions)

#### 4.1.2 Codex Message Processor

- **Function**: Serves as the **translation layer** between the client protocol and Codex core internals.
- **Inbound path**: Translates JSON-RPC requests into Codex core operations (e.g., `turn/start` → core turn initiation).
- **Outbound path**: Listens to Codex core's internal event stream, transforms low-level internal events into a **stable, UI-ready set of JSON-RPC notifications**.

This translation layer is architecturally critical. Codex core's internal event model is rich and implementation-specific; the App Server's notification set is a curated, stable projection designed for UI consumption. This decoupling allows Codex core to evolve its internal representations without breaking client contracts.

#### 4.1.3 Thread Manager

- **Function**: Maintains a registry of active core threads, identified by thread IDs.
- **Lifecycle management**: Spins up one Codex core session per thread, routes messages to the correct session, and handles session teardown.
- **Concurrency model**: Supports multiple concurrent threads within a single App Server process, enabling use cases like the Desktop App's parallel agent orchestration.

#### 4.1.4 Core Threads

- **Function**: Each core thread is an instantiation of the Codex core runtime, executing the agent loop for a single conversation.
- **Isolation**: Core threads are logically isolated; one thread's state, history, and tool execution context do not leak into another.

### 4.2 Message Flow Dynamics

The relationship between client requests and server notifications is fundamentally **one-to-many**:

$$
\text{ClientRequest}(r) \rightarrow \{ \text{Notification}_1, \text{Notification}_2, \ldots, \text{Notification}_k \}
$$

where $k$ varies depending on the complexity of the agent's response. A single user message may trigger:

1. Turn started notification
2. User message item (started → completed)
3. Multiple tool call items (started → approval request → approval response → completed)
4. Agent message item (started → delta₁ → delta₂ → ... → deltaₙ → completed)
5. Turn completed notification

This one-to-many cardinality is what makes request/response protocols (like REST) fundamentally unsuitable for this use case.

---

## 5. JSON-RPC Protocol Design and Bidirectional Communication

### 5.1 Why JSON-RPC 2.0

JSON-RPC 2.0 was selected for the following properties:

| Property | Benefit |
|----------|---------|
| **Bidirectional** | Both client and server can send requests (critical for approval gates) |
| **Transport-agnostic** | Works over stdio, WebSocket, HTTP—no transport coupling |
| **Lightweight** | Minimal framing overhead compared to gRPC/protobuf for a single-connection scenario |
| **Language-neutral** | JSON parsing is universally available; no code generation required for basic integration |
| **Notification support** | Fire-and-forget messages (no response expected) map naturally to streaming events |
| **Explicit error model** | Structured error codes and messages for robust client error handling |

### 5.2 Transport Layer: JSONL over stdio

Each JSON-RPC message is serialized as a single line of JSON terminated by `\n`. This gives:

- **Framing**: Newline delimitation provides unambiguous message boundaries without length-prefix headers.
- **Debuggability**: Messages are human-readable in raw pipe output.
- **Streaming compatibility**: Line-buffered I/O works naturally with OS pipe semantics.

### 5.3 Bidirectional Request/Response Symmetry

The protocol supports four message patterns:

| Pattern | Direction | Example |
|---------|-----------|---------|
| **Client → Server Request** | Client initiates, server responds | `initialize`, `turn/start`, `thread/start` |
| **Server → Client Notification** | Server pushes, no response expected | `item/started`, `item/*/delta`, `turn/completed` |
| **Server → Client Request** | Server initiates, client responds | `item/commandExecution/requestApproval` |
| **Client → Server Notification** | Client pushes, no response expected | (reserved for future use) |

The **server-initiated request** pattern is particularly noteworthy. When the agent determines it needs to execute a command that requires human approval, the server sends a JSON-RPC request to the client with a unique `id`. The agent loop **pauses** until the client responds with either `"allow"` or `"deny"`. This transforms a fundamentally asynchronous agent loop into a controlled, human-in-the-loop system without polling or callback URLs.

### 5.4 Schema Generation and Type Safety

To support multi-language client development without maintaining hand-written bindings, the App Server provides two schema generation commands:

**TypeScript definitions** (generated directly from Rust protocol types):

```bash
codex app-server generate-ts
```

**JSON Schema bundle** (for feeding into arbitrary code generators):

```bash
codex app-server generate-json-schema
```

This approach ensures that the Rust source types are the **single source of truth**, and all client-side type definitions are derived rather than duplicated.

---

## 6. Conversation Primitives: Formal Specification

The App Server protocol is built on three hierarchical primitives that provide a complete, composable grammar for representing agent interactions.

### 6.1 Item

An **item** $I$ is the atomic unit of input/output in the Codex protocol. Formally:

$$
I = (\text{id}, \tau, \mathcal{L})
$$

where:

- $\text{id}$ is a unique identifier within the thread
- $\tau \in \{\texttt{user\_message}, \texttt{agent\_message}, \texttt{tool\_execution}, \texttt{approval\_request}, \texttt{diff}, \ldots\}$ is the item type
- $\mathcal{L}$ is the lifecycle state machine

#### 6.1.1 Item Lifecycle State Machine

Every item follows an explicit finite-state lifecycle:

$$
\texttt{item/started} \xrightarrow[\text{(optional, repeated)}]{} \texttt{item/*/delta} \xrightarrow{} \texttt{item/completed}
$$

More precisely:

```
                    ┌──────────────────────┐
                    │                      │
                    ▼                      │
   item/started ──────▶ item/*/delta ──────┘
                    │
                    ▼
              item/completed
```

| Event | Semantics | Client Action |
|-------|-----------|---------------|
| `item/started` | Item begins; type and metadata are available | Allocate UI element, begin rendering placeholder |
| `item/*/delta` | Incremental content update (e.g., text chunk, partial diff) | Append to existing rendering |
| `item/completed` | Item finalized; terminal payload available | Finalize rendering, commit to local state |

The explicit lifecycle has three critical engineering benefits:

1. **Immediate rendering**: Clients can begin rendering on `started` without waiting for the full payload.
2. **Streaming compatibility**: Delta events enable character-by-character or chunk-by-chunk display of agent reasoning.
3. **Deterministic finalization**: `completed` provides an unambiguous signal that no more deltas will arrive for this item.

### 6.2 Turn

A **turn** $t$ is one unit of agent work initiated by user input:

$$
t = (\text{input}, \langle I_1, I_2, \ldots, I_m \rangle)
$$

A turn begins when the client submits an input via `turn/start` and ends when the server emits `turn/completed`. The sequence of items within a turn represents the complete intermediate and final outputs produced by the agent in response to that input.

**Key property**: A turn is a **transactional boundary**. The client can reason about the agent's work in terms of complete turns, and the turn boundary is the natural point for:

- Displaying "agent is working" / "agent is done" indicators
- Committing accumulated diffs to the workspace
- Updating token/cost accounting

### 6.3 Thread

A **thread** $\mathcal{T}$ is the durable container:

$$
\mathcal{T} = (\text{id}, \text{state}, \langle t_1, t_2, \ldots, t_n \rangle, \mathcal{H})
$$

where $\text{state} \in \{\texttt{active}, \texttt{archived}\}$ and $\mathcal{H}$ is the persisted event history.

Thread operations and their semantics:

| Operation | Input | Output | Side Effect |
|-----------|-------|--------|-------------|
| `create` | Configuration | Thread ID | New empty thread instantiated |
| `resume` | Thread ID | Event history replay | Client receives full history to reconstruct UI |
| `fork` | Thread ID, turn index $k$ | New Thread ID | New thread with $\langle t_1, \ldots, t_k \rangle$ copied |
| `archive` | Thread ID | Confirmation | Thread marked inactive, history preserved |

### 6.4 Hierarchical Composition

The three primitives compose into a clean hierarchy:

$$
\underbrace{\mathcal{T}}_{\text{Thread}} \supset \underbrace{t_1, t_2, \ldots, t_n}_{\text{Turns}} \supset \underbrace{I_{i,1}, I_{i,2}, \ldots, I_{i,m_i}}_{\text{Items}}
$$

This hierarchy is both a **data model** (for persistence) and an **event stream grammar** (for real-time rendering). Any well-formed Codex session is a valid instantiation of this hierarchy, and any conforming client can render it by processing the event stream in order.

---

## 7. Protocol Message Flow: Detailed Walkthrough

### 7.1 Phase 1: Initialization Handshake

The initialization handshake must be the **first exchange** before any operational methods are invoked.

**Client → Server:**

```json
{
  "jsonrpc": "2.0",
  "method": "initialize",
  "id": 0,
  "params": {
    "clientInfo": {
      "name": "codex_vscode",
      "title": "Codex VS Code Extension",
      "version": "0.1.0"
    }
  }
}
```

**Server → Client:**

```json
{
  "jsonrpc": "2.0",
  "id": 0,
  "result": {
    "userAgent": "codex_vscode/0.94.0-alpha.7 (Mac OS 26.2.0; arm64) vscode/2.4.22 (codex_vscode; 0.1.0)"
  }
}
```

**Purpose of the handshake:**

- **Capability negotiation**: The server can advertise supported features, enabling clients to degrade gracefully when connecting to older servers.
- **Protocol versioning**: Both sides agree on the protocol version, enabling backward-compatible evolution.
- **Feature flags**: Runtime feature toggles can be communicated without protocol changes.
- **Client identification**: The server can tailor behavior (e.g., logging, telemetry) based on client identity.

### 7.2 Phase 2: Thread and Turn Lifecycle

**Client → Server:**

```json
{"jsonrpc": "2.0", "method": "thread/start", "id": 1, "params": {}}
```

```json
{"jsonrpc": "2.0", "method": "turn/start", "id": 2, "params": {"message": "run tests and summarize failures"}}
```

**Server → Client (notification stream):**

```json
{"jsonrpc": "2.0", "method": "thread/started", "params": {"threadId": "th_abc123"}}
{"jsonrpc": "2.0", "method": "turn/started", "params": {"turnId": "tn_001"}}
{"jsonrpc": "2.0", "method": "item/started", "params": {"itemId": "it_001", "type": "user_message"}}
{"jsonrpc": "2.0", "method": "item/completed", "params": {"itemId": "it_001", "content": "run tests and summarize failures"}}
```

### 7.3 Phase 3: Tool Execution with Approval Gate

**Server → Client (request):**

```json
{
  "jsonrpc": "2.0",
  "method": "item/commandExecution/requestApproval",
  "id": 100,
  "params": {
    "itemId": "it_002",
    "command": "pnpm test",
    "reason": "Run the project's test suite to identify failures"
  }
}
```

**Client → Server (response):**

```json
{
  "jsonrpc": "2.0",
  "id": 100,
  "result": {
    "decision": "allow",
    "remember": true
  }
}
```

The `remember` field enables **progressive trust**: the client can instruct the server to auto-approve future commands matching a pattern (e.g., commands starting with `pnpm test`), reducing friction over time while maintaining the initial safety gate.

**Server → Client (notification, post-approval):**

```json
{"jsonrpc": "2.0", "method": "item/completed", "params": {"itemId": "it_002", "type": "tool_execution", "command": "pnpm test", "exitCode": 1, "stdout": "...", "stderr": "..."}}
```

### 7.4 Phase 4: Streaming Agent Response

**Server → Client (notification stream):**

```json
{"jsonrpc": "2.0", "method": "item/started", "params": {"itemId": "it_003", "type": "agent_message"}}
{"jsonrpc": "2.0", "method": "item/agentMessage/delta", "params": {"itemId": "it_003", "delta": "Ran 3 tests. "}}
{"jsonrpc": "2.0", "method": "item/agentMessage/delta", "params": {"itemId": "it_003", "delta": "2 passed, 1 failed. The failing test is..."}}
{"jsonrpc": "2.0", "method": "item/completed", "params": {"itemId": "it_003", "content": "Ran 3 tests. 2 passed, 1 failed. The failing test is..."}}
{"jsonrpc": "2.0", "method": "turn/completed", "params": {"turnId": "tn_001"}}
```

### 7.5 Debug and Inspection

The full message exchange for any turn can be captured using the built-in debug client:

```bash
codex debug app-server send-message-v2 "run tests and summarize failures"
```

This command launches an App Server process, performs the initialization handshake, sends the specified message as a turn, and prints all JSON-RPC messages exchanged—providing a complete protocol trace for debugging and integration testing.

---

## 8. Multi-Surface Client Integration Patterns

### 8.1 Pattern 1: Local Apps and IDEs

**Applicable surfaces**: VS Code extension, JetBrains IDEs, Xcode, macOS Desktop App

**Architecture:**

```
┌──────────────────────┐
│   IDE / Desktop App  │
│   ┌──────────────┐   │
│   │  JSON-RPC    │   │
│   │  Client      │   │
│   └──────┬───────┘   │
│          │ stdio     │
│   ┌──────▼───────┐   │
│   │  App Server  │   │
│   │  (child proc)│   │
│   └──────────────┘   │
└──────────────────────┘
```

**Integration mechanics:**

1. The client application **bundles or fetches** a platform-specific App Server binary (e.g., `codex-aarch64-apple-darwin`).
2. The binary is launched as a **child process** with stdin/stdout pipes connected.
3. The client maintains a **bidirectional JSONL channel** for the lifetime of the session.
4. The shipped binary is **pinned to a tested version**, ensuring deterministic behavior.

**Release cycle decoupling:**

Some integrations (e.g., Xcode) cannot ship client updates frequently. These partners adopt a **split-release strategy**:

- The **client** remains stable across releases.
- The **App Server binary** can be updated independently (e.g., via a background download).
- Backward compatibility of the JSON-RPC protocol ensures that older clients work with newer servers.

This decoupling is formally expressed as a **compatibility invariant**:

$$
\forall\, c \in \text{ClientVersions},\, s \in \text{ServerVersions}: v(c) \leq v(s) \implies \text{Compatible}(c, s)
$$

where $v(\cdot)$ returns the protocol version. Newer servers must accept all messages from older clients.

**Concurrency model (Desktop App):**

The macOS Desktop App orchestrates **multiple concurrent Codex agents**. Each agent corresponds to a separate thread within a single App Server process, leveraging the Thread Manager's concurrent session support:

$$
\text{DesktopApp} \xrightarrow{\text{thread/start} \times N} \text{AppServer} \xrightarrow{\text{spawn}} \{\text{CoreThread}_1, \ldots, \text{CoreThread}_N\}
$$

### 8.2 Pattern 2: Codex Web Runtime

**Applicable surfaces**: Codex web application (browser-based)

**Architecture:**

```
┌──────────┐      ┌──────────────┐      ┌───────────────────────┐
│  Browser │ HTTP │  Codex       │ stdio│  Container            │
│  Tab     │◄────▶│  Backend     │◄────▶│  ┌─────────────────┐  │
│  (SSE)   │  SSE │  (Worker)    │      │  │  App Server     │  │
└──────────┘      └──────────────┘      │  │  (binary)       │  │
                                        │  └─────────────────┘  │
                                        │  ┌─────────────────┐  │
                                        │  │  Workspace       │  │
                                        │  │  (checked out)   │  │
                                        │  └─────────────────┘  │
                                        └───────────────────────┘
```

**Key design decisions:**

1. **Server-side state ownership**: The browser is an **ephemeral client**—tabs close, networks drop. The Codex backend (worker) maintains the JSON-RPC session with the App Server in the container. Task progress persists regardless of browser connectivity.

2. **Transport bridging**: The worker bridges between HTTP/SSE (browser-facing) and JSON-RPC over stdio (App Server-facing). This keeps the browser-side implementation lightweight (SSE consumer) while preserving the full protocol richness internally.

3. **Reconnection semantics**: When a new browser session connects to an in-progress task, the backend replays the thread's persisted event history, allowing the client to reconstruct the full UI state without the App Server needing to re-execute anything.

4. **Container isolation**: Each web task runs in a provisioned container with the checked-out workspace, providing filesystem isolation and reproducible environments.

### 8.3 Pattern 3: TUI/CLI (Planned Migration)

**Current state**: The TUI is a "native" client that runs **in-process** with the Codex core runtime, directly invoking Rust types without the JSON-RPC protocol layer.

**Planned state**: Refactoring the TUI to use the App Server protocol, making it architecturally identical to any other client:

```
┌──────────┐  stdio  ┌──────────────┐
│   TUI    │◄──────▶│  App Server  │
│  (client)│         │  (child proc)│
└──────────┘         └──────────────┘
```

**Benefits of migration:**

| Benefit | Description |
|---------|-------------|
| **Architectural consistency** | TUI becomes a standard client; no special-case code paths in Codex core |
| **Remote execution** | TUI can connect to an App Server running on a remote machine (e.g., cloud workstation), keeping compute close to the codebase |
| **Session persistence** | Laptop can sleep/disconnect; App Server continues running remotely, and TUI reconnects to resume |
| **Protocol coverage** | TUI exercises the same protocol paths as all other clients, improving test coverage |

---

## 9. Protocol Comparison and Selection Framework

### 9.1 Decision Matrix

| Criterion | App Server (JSON-RPC) | MCP Server | Cross-Provider Harness | Codex Exec (CLI) | Codex SDK (TypeScript) |
|-----------|----------------------|-----------|----------------------|-------------------|----------------------|
| **Full agent loop** | ✅ Complete | ⚠️ Subset | ⚠️ Common denominator | ⚠️ Non-interactive | ✅ Complete |
| **Streaming items** | ✅ Delta events | ❌ Request/response | ⚠️ Varies | ✅ Structured output | ✅ Callbacks |
| **Approval gates** | ✅ Server-initiated requests | ❌ Not supported | ⚠️ Varies | ❌ Non-interactive | ✅ Supported |
| **Thread persistence** | ✅ Full lifecycle | ❌ Stateless | ⚠️ Varies | ❌ Ephemeral | ⚠️ Limited |
| **Auth integration** | ✅ Sign in with ChatGPT | ❌ External | ❌ External | ✅ CLI auth | ⚠️ Manual |
| **Multi-language support** | ✅ JSON Schema codegen | ✅ MCP SDKs | ⚠️ Varies | ✅ Any shell | ❌ TypeScript only |
| **Integration effort** | Medium (JSON-RPC binding) | Low (MCP client) | Medium | Very low (shell) | Low (npm install) |
| **Backward compatibility** | ✅ Designed for | ⚠️ MCP-dependent | ⚠️ Provider-dependent | N/A | ⚠️ SDK version |

### 9.2 Recommended Selection Logic

$$
\text{Protocol}^* = \arg\max_{\text{protocol}} \sum_{i} w_i \cdot \text{Score}_i(\text{protocol})
$$

where $w_i$ are weights reflecting the integration's priorities (e.g., interactivity, streaming, persistence).

**Practical guidance:**

- **Default recommendation**: Use the **App Server** for any integration requiring interactive, streaming, or multi-turn agent experiences.
- **MCP Server** (`codex mcp-server`): Use when integrating into an existing MCP-based orchestration framework (e.g., OpenAI Agents SDK) and the interaction is primarily tool-invocation-style.
- **Cross-provider harness protocols**: Use when the integration must remain provider-agnostic across multiple LLM backends. Accept the capability ceiling imposed by the common denominator.
- **Codex Exec**: Use for CI/CD, automation pipelines, and batch processing where the interaction is non-interactive (fire-and-forget).
- **Codex SDK**: Use for TypeScript server-side applications that want a native library interface without managing JSON-RPC bindings.

### 9.3 MCP Server Integration Details

```bash
codex mcp-server
```

This command starts Codex as an MCP-compliant tool server, exposable to any MCP client via stdio. While functional, the mapping limitations include:

- **No streaming agent messages**: MCP tool calls return complete results, not incremental deltas.
- **No server-initiated requests**: Approval gates cannot be represented in the MCP tool-call model.
- **No thread lifecycle**: Each MCP invocation is stateless; multi-turn context must be managed by the MCP host.

These limitations are not bugs—they are inherent to MCP's design as a **tool-invocation protocol**, not a **session protocol**.

---

## 10. Supplementary Embedding Mechanisms

### 10.1 Codex Exec

```bash
codex exec "run tests and fix failures" --output structured
```

**Characteristics:**

- Single command, runs to completion non-interactively
- Streams structured output (JSON events) to stdout for log ingestion
- Exits with a clear success/failure exit code
- Ideal for CI/CD integration, automated code review pipelines, and batch processing

**Use case example (CI pipeline):**

```yaml
# GitHub Actions
- name: Run Codex code review
  run: codex exec "review the PR diff and flag potential issues" --output json > review.json
```

### 10.2 Codex SDK

```typescript
import { CodexAgent } from "@openai/codex-sdk";

const agent = new CodexAgent({ model: "gpt-5.2-codex" });
const result = await agent.run("run tests and summarize failures", {
  workingDirectory: "./my-project",
  onItem: (item) => console.log(item),
});
```

**Characteristics:**

- TypeScript-native library interface
- Programmatic control of local Codex agents
- Callback-based streaming of items
- Shipped earlier than the App Server; currently covers a smaller surface area
- Future SDKs may wrap the App Server protocol for additional language support

---

## 11. Systems Engineering Considerations

### 11.1 Process Lifecycle Management

The App Server's child-process model introduces specific lifecycle considerations:

**Startup ordering:**

$$
\text{Client starts} \xrightarrow{\text{spawn}} \text{App Server process} \xrightarrow{\text{stdio ready}} \text{Initialize handshake} \xrightarrow{\text{ack}} \text{Operational}
$$

**Shutdown semantics:**

- **Graceful**: Client sends a shutdown request; App Server persists thread state and exits.
- **Ungraceful (client death)**: App Server detects stdin EOF (pipe closure from parent death) and initiates self-shutdown with state persistence.
- **Ungraceful (server death)**: Client detects stdout EOF and can restart the App Server, reconnecting via thread resume.

### 11.2 Backward Compatibility Engineering

The protocol's backward compatibility guarantee is maintained through several mechanisms:

1. **Additive-only evolution**: New methods and notification types can be added; existing ones are never removed or have their semantics changed.
2. **Unknown field tolerance**: Clients must ignore unknown fields in JSON objects (forward compatibility).
3. **Capability advertisement**: The `initialize` response includes capability flags, enabling feature detection.
4. **Version negotiation**: Protocol version can be negotiated during initialization for major breaking changes (reserved for extreme cases).

### 11.3 Concurrency and Resource Management

For the Desktop App's parallel agent orchestration:

$$
\text{Resource}(\text{AppServer}) = O(N) \cdot \text{Resource}(\text{CoreThread})
$$

where $N$ is the number of concurrent threads. Each core thread maintains:

- An independent model inference context
- Separate tool execution sandbox
- Isolated thread history

The Thread Manager provides fair scheduling and resource accounting across concurrent threads.

### 11.4 Security Model

The App Server inherits the Codex harness's security model:

- **Sandboxed tool execution**: Shell commands run in controlled environments with explicit filesystem access policies.
- **Approval gates**: Human-in-the-loop confirmation for sensitive operations.
- **Credential isolation**: Authentication tokens are managed by the App Server process and not exposed to tool execution environments.
- **Progressive trust**: The `remember` field in approval responses enables per-session trust escalation without global policy changes.

---

## 12. Formal Analysis and Design Rationale

### 12.1 Protocol Expressiveness

The App Server protocol's expressiveness can be characterized by its ability to represent the full space of agent interactions. Let $\mathcal{A}$ be the set of all possible agent behaviors. The protocol's coverage is:

$$
\text{Coverage}(\text{protocol}) = \frac{|\mathcal{A}_{\text{representable}}|}{|\mathcal{A}|}
$$

The App Server achieves near-complete coverage because:

- **Items are typed and extensible**: New item types can be introduced without protocol changes.
- **Lifecycle events are universal**: The started/delta/completed pattern applies uniformly to all item types.
- **Bidirectional requests handle all control flow**: Approval gates, configuration prompts, and any future human-in-the-loop interactions are naturally expressed.

Compare with MCP:

$$
\text{Coverage}(\text{MCP}) < \text{Coverage}(\text{AppServer})
$$

because MCP cannot represent streaming items, server-initiated requests, or stateful thread semantics.

### 12.2 State Machine Formalization

The complete thread lifecycle can be formalized as a hierarchical state machine:

$$
\text{Thread}: \quad \texttt{created} \xrightarrow{\texttt{start}} \texttt{active} \xrightarrow{\texttt{archive}} \texttt{archived}
$$

$$
\text{Turn (within active thread)}: \quad \texttt{idle} \xrightarrow{\texttt{turn/start}} \texttt{running} \xrightarrow{\texttt{turn/completed}} \texttt{idle}
$$

$$
\text{Item (within running turn)}: \quad \texttt{pending} \xrightarrow{\texttt{started}} \texttt{streaming} \xrightarrow{\texttt{completed}} \texttt{finalized}
$$

$$
\text{Approval (within streaming item)}: \quad \texttt{executing} \xrightarrow{\texttt{requestApproval}} \texttt{awaiting} \xrightarrow{\texttt{allow|deny}} \texttt{resolved}
$$

Each state transition corresponds to exactly one JSON-RPC message, ensuring **protocol-state isomorphism**: the sequence of messages uniquely determines the system state, and vice versa.

### 12.3 Consistency Under Disconnection

For the web runtime, where browser disconnections are routine, the following invariant must hold:

$$
\text{State}(\text{reconnected client}) \equiv \text{State}(\text{continuous client})
$$

This is achieved by making the **persisted event history the sole source of truth**. Client UI state is always a deterministic function of the event history:

$$
\text{UIState} = f(\mathcal{H})
$$

where $f$ is a pure function that processes the event history $\mathcal{H}$ in order. Reconnection simply involves replaying $\mathcal{H}$ through $f$.

### 12.4 Information-Theoretic Efficiency of the Delta Streaming Model

The delta streaming model for items is information-theoretically efficient. Consider an agent message of total length $L$ tokens. Without streaming, the client waits for all $L$ tokens before rendering, introducing latency:

$$
\text{Latency}_{\text{batch}} = L \cdot \bar{t}_{\text{token}}
$$

where $\bar{t}_{\text{token}}$ is the average per-token generation time. With delta streaming, the client begins rendering after the first delta:

$$
\text{Latency}_{\text{stream}} = \bar{t}_{\text{token}} + t_{\text{transport}}
$$

The perceived responsiveness improvement is:

$$
\Delta_{\text{responsiveness}} = (L - 1) \cdot \bar{t}_{\text{token}} - t_{\text{transport}}
$$

For typical values ($L \sim 500$, $\bar{t}_{\text{token}} \sim 20\text{ms}$, $t_{\text{transport}} \sim 1\text{ms}$), this represents approximately **10 seconds** of perceived latency reduction—a transformative UX improvement.

---

## 13. Conclusion and Forward Trajectory

### 13.1 Summary of Contributions

The Codex App Server represents a principled solution to the agent-surface unification problem. Its key contributions are:

1. **Architectural decoupling**: The translation layer between Codex core's internal event model and the stable JSON-RPC protocol surface allows both to evolve independently.

2. **Protocol design**: The three-primitive grammar (Thread → Turn → Item) with explicit lifecycle state machines provides a complete, composable, and extensible representation of agent interactions.

3. **Bidirectional communication**: Server-initiated requests elegantly solve the human-in-the-loop approval problem without polling or callbacks.

4. **Multi-surface integration**: A single protocol supports five distinct client architectures (CLI, IDE, web, desktop, third-party), with well-defined integration patterns for each.

5. **Backward compatibility by design**: Additive-only evolution, unknown field tolerance, and capability advertisement ensure long-term protocol stability.

### 13.2 Forward Directions

| Direction | Description |
|-----------|-------------|
| **TUI migration** | Refactoring the TUI to use the App Server protocol, enabling remote execution and session persistence |
| **Additional language SDKs** | Wrapping the App Server protocol in language-native SDKs (Python, Go, Swift, Kotlin) to reduce integration friction |
| **Skills ecosystem** | Expanding the skills framework as a common standard for extending agent capabilities across providers |
| **Protocol standardization** | Contributing protocol learnings to emerging cross-provider agent harness standards |

### 13.3 Availability

All source code is available in the **Codex CLI open-source repository**:

- Protocol definitions: Rust types in `codex-core`
- App Server implementation: `codex-app-server` crate
- TypeScript schema generation: `codex app-server generate-ts`
- JSON Schema generation: `codex app-server generate-json-schema`
- Debug client: `codex debug app-server send-message-v2`

---

*This report constitutes a comprehensive technical analysis of the Codex App Server architecture, protocol design, and integration patterns as deployed across OpenAI's production surfaces and partner integrations.*